# Notebook 04 — Construir indice vectorial

Objetivo de esta fase: tomar los 389 documentos `.md`, dividirlos en chunks, calcular embeddings con OpenAI y persistirlos en dos colecciones Chroma (`mundial` y `plataforma`) que el agente Agentic RAG consultará.

## Decisiones tomadas

| Item | Decision | Razon |
|---|---|---|
| Modelo embeddings | `text-embedding-3-small` (OpenAI) | 1536-dim, multilingual de calidad, simple. Reemplaza la version previa con `multilingual-e5-base` que causaba crash del kernel en Windows por DLL conflict torch+chromadb (`0xC0000005`). Costo: ~$0.10 indexado completo, ~$0.0001 por query. Token del curso lo cubre. |
| Chunking | `MarkdownHeaderTextSplitter` → fallback `RecursiveCharacterTextSplitter(2000, 200)` | Aprovecha la estructura semantica explicita de los `.md` (headers `#`/`##`/`###`). 2000/200 chars ≈ 500/50 tokens (target Plan §7). |
| Filtro chunks | min 40 chars | Descarta headers huerfanos sin texto. |
| Colecciones | Dos separadas: `mundial` y `plataforma` | Alineado con arquitectura Plan §3 — agente ReAct elige tool segun pregunta. |
| Distancia | Cosine | Estandar para embeddings normalizados. |
| Persistencia | `./chroma_db/` (relativa al proyecto) | Reutilizable en F6 sin recomputar. |


## 1. Imports y configuracion

In [1]:
from __future__ import annotations

import os
import re
import time
from pathlib import Path

import frontmatter
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from tqdm.auto import tqdm

import chromadb

# .env (OPENAI_API_KEY)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en .env"

# Rutas
CORPUS_91 = PROJECT_ROOT / "Corpus_91"
CORPUS_MUNDIAL = PROJECT_ROOT / "Corpus_Mundial"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"

# Modelo / chunking
EMBEDDING_MODEL = "text-embedding-3-small"
# Plan §7: ~500 tokens / 50 overlap. En caracteres ≈ 2000/200 para español
# (un token ~4 chars con tokenizer subword).
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200
MAX_CHUNK_CHARS = 2400   # ~600 tokens, da margen al overlap
MIN_CHUNK_CHARS = 40     # descarta headers huerfanos
BATCH_SIZE_EMBED = 256   # OpenAI acepta hasta 2048 strings por request
BATCH_SIZE_CHROMA = 256

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Corpus_91 existe:      {CORPUS_91.exists()}")
print(f"Corpus_Mundial existe: {CORPUS_MUNDIAL.exists()}")

Embedding model: text-embedding-3-small
Project root: c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final
Corpus_91 existe:      True
Corpus_Mundial existe: True


## 2. Loader de documentos

Recorre cada corpus, parsea frontmatter YAML, construye lista de dicts con contenido + metadata. Sanitiza metadata para que sea compatible con Chroma (solo `str`/`int`/`float`/`bool`; listas se serializan como `"a, b, c"`).

In [2]:
def sanitize_metadata(meta: dict) -> dict:
    """Chroma solo acepta primitivos. Listas → 'a, b, c'. None → ''. Resto → str."""
    out = {}
    for k, v in meta.items():
        if v is None:
            out[k] = ""
        elif isinstance(v, (str, int, float, bool)):
            out[k] = v
        elif isinstance(v, (list, tuple)):
            out[k] = ", ".join(str(x) for x in v)
        else:
            out[k] = str(v)
    return out


def load_corpus(root: Path, coleccion: str) -> list[dict]:
    """Carga recursivamente todos los .md bajo `root` y agrega metadata `coleccion`."""
    docs = []
    for path in sorted(root.rglob("*.md")):
        if path.name.lower() == "readme.md":
            continue
        try:
            post = frontmatter.load(path)
        except Exception as e:
            print(f"[skip] {path.relative_to(root)} — error parse: {e}")
            continue
        meta = dict(post.metadata)
        meta["coleccion"] = coleccion
        meta["ruta"] = str(path.relative_to(PROJECT_ROOT).as_posix())
        rel_parts = path.relative_to(root).parts
        meta["categoria"] = rel_parts[0] if len(rel_parts) > 1 else ""
        docs.append({
            "content": post.content.strip(),
            "metadata": sanitize_metadata(meta),
        })
    return docs


docs_plataforma = load_corpus(CORPUS_91, "plataforma")
docs_mundial = load_corpus(CORPUS_MUNDIAL, "mundial")

print(f"Corpus_91 (plataforma): {len(docs_plataforma)} docs")
print(f"Corpus_Mundial:         {len(docs_mundial)} docs")
print(f"TOTAL: {len(docs_plataforma) + len(docs_mundial)} docs")

print("\nEjemplo metadata Corpus_91:")
print(docs_plataforma[0]["metadata"])
print("\nEjemplo metadata Corpus_Mundial:")
print(docs_mundial[0]["metadata"])

Corpus_91 (plataforma): 55 docs
Corpus_Mundial:         438 docs
TOTAL: 493 docs

Ejemplo metadata Corpus_91:
{'titulo': 'Qué es 91', 'tema': 'identidad', 'tipo': 'concepto', 'fuente': 'General.txt', 'tags': 'overview, plataforma, mundial-2026, no-apuestas', 'coleccion': 'plataforma', 'ruta': 'Corpus_91/identidad/plataforma-overview.md', 'categoria': 'identidad'}

Ejemplo metadata Corpus_Mundial:
{'titulo': 'Mexico vs Sudafrica — Fase de Grupos, Grupo A', 'tema': 'calendario-mundial-2026', 'tipo': 'partido', 'fuente': 'Plataforma 91 / FIFA', 'partido_id': 1, 'edicion': 2026, 'fase': 'group', 'fase_es': 'Fase de Grupos', 'grupo': 'A', 'fecha': '2026-06-11', 'hora_local': '14:00', 'hora_utc': '19:00', 'estadio': 'Estadio Azteca', 'ciudad': 'Mexico City', 'pais': 'Mexico', 'equipo_local': 'Mexico', 'equipo_visitante': 'Sudafrica', 'equipo_local_original': 'Mexico', 'equipo_visitante_original': 'South Africa', 'tags': 'mundial-2026, calendario, partido, group, mexico, sudafrica, estadio-az

## 3. Chunking — splitters en Python puro

Pipeline en dos pasos, implementado sin `langchain_text_splitters` para evitar crash del kernel en Windows (esa libreria carga `tokenizers` Rust con DLL conflict).

1. **`split_by_markdown_headers`** corta por headers `#`, `##`, `###`. Aprovecha que cada doc tiene secciones bien delimitadas (introduccion + secciones + FAQs). Inyecta el header padre al inicio de cada sub-chunk para preservar contexto.
2. **`recursive_split`** se aplica a cualquier sub-chunk que siga siendo grande (>2400 chars). Intenta separadores en orden: `\n\n`, `\n`, `. `, ` `, char. Equivalente funcional al `RecursiveCharacterTextSplitter` de LangChain.

Tras chunking se descartan trozos <40 chars (headers huerfanos). Cada chunk hereda la metadata del doc original mas: `header_h1`/`h2`/`h3` (si aplica) y `chunk_index`.

In [3]:
HEADER_REGEX = re.compile(r"^(#{1,3})\s+(.*)$", re.MULTILINE)
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]


def split_by_markdown_headers(text):
    # Devuelve lista de (header_path, content). header_path es dict h1/h2/h3.
    # Si no hay headers, devuelve [({}, text)] entero.
    matches = list(HEADER_REGEX.finditer(text))
    if not matches:
        return [({}, text)]

    sections = []
    current = {"h1": "", "h2": "", "h3": ""}
    # Preambulo antes del primer header
    if matches[0].start() > 0:
        pre = text[: matches[0].start()].strip()
        if pre:
            sections.append((dict(current), pre))

    for i, m in enumerate(matches):
        level = len(m.group(1))
        title = m.group(2).strip()
        # Actualizar path de headers
        if level == 1:
            current = {"h1": title, "h2": "", "h3": ""}
        elif level == 2:
            current["h2"] = title
            current["h3"] = ""
        elif level == 3:
            current["h3"] = title
        # Cuerpo desde m hasta el siguiente header (o EOF)
        body_start = m.start()
        body_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[body_start:body_end].strip()
        if body:
            sections.append((dict(current), body))
    return sections


def recursive_split(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, separators=SEPARATORS):
    # Equivalente a RecursiveCharacterTextSplitter: prueba separadores en orden
    # y mantiene overlap entre chunks.
    if len(text) <= chunk_size:
        return [text]

    # Encontrar primer separador presente en el texto
    sep = ""
    for s in separators:
        if s == "":
            sep = ""
            break
        if s in text:
            sep = s
            break

    if sep == "":
        # No hay separadores; hard split por chunk_size con overlap
        chunks = []
        i = 0
        step = max(chunk_size - overlap, 1)
        while i < len(text):
            chunks.append(text[i: i + chunk_size])
            i += step
        return chunks

    # Split por el separador y reagrupar respetando chunk_size + overlap
    parts = text.split(sep)
    chunks = []
    buf = ""
    for p in parts:
        candidate = (buf + sep + p) if buf else p
        if len(candidate) <= chunk_size:
            buf = candidate
        else:
            if buf:
                chunks.append(buf)
            # Si la parte sola es muy grande, recursar con siguiente separador
            if len(p) > chunk_size:
                sub_seps = separators[separators.index(sep) + 1:] if sep in separators else [""]
                chunks.extend(recursive_split(p, chunk_size, overlap, sub_seps))
                buf = ""
            else:
                buf = p
    if buf:
        chunks.append(buf)

    # Aplicar overlap: cada chunk (excepto el primero) toma N chars del anterior
    if overlap > 0 and len(chunks) > 1:
        with_overlap = [chunks[0]]
        for i in range(1, len(chunks)):
            prev_tail = chunks[i - 1][-overlap:]
            with_overlap.append(prev_tail + chunks[i])
        chunks = with_overlap

    return chunks


def chunk_document(doc):
    content = doc["content"]
    base_meta = doc["metadata"]
    chunks_out = []

    sections = split_by_markdown_headers(content)
    for headers, text in sections:
        merged = dict(base_meta)
        if headers.get("h1"):
            merged["header_h1"] = headers["h1"]
        if headers.get("h2"):
            merged["header_h2"] = headers["h2"]
        if headers.get("h3"):
            merged["header_h3"] = headers["h3"]

        if len(text) > MAX_CHUNK_CHARS:
            for sub in recursive_split(text):
                chunks_out.append({"content": sub, "metadata": dict(merged)})
        else:
            chunks_out.append({"content": text, "metadata": dict(merged)})

    # Filtrar chunks vacios/chiquitos
    chunks_out = [c for c in chunks_out if len(c["content"].strip()) >= MIN_CHUNK_CHARS]
    for i, c in enumerate(chunks_out):
        c["metadata"]["chunk_index"] = i
    return chunks_out


def chunk_corpus(docs):
    all_chunks = []
    for d in docs:
        all_chunks.extend(chunk_document(d))
    return all_chunks


t0 = time.time()
chunks_plataforma = chunk_corpus(docs_plataforma)
chunks_mundial = chunk_corpus(docs_mundial)
print(f"Chunking en {time.time() - t0:.1f}s")

print(f"\nChunks plataforma: {len(chunks_plataforma)}")
print(f"Chunks mundial:    {len(chunks_mundial)}")
print(f"TOTAL: {len(chunks_plataforma) + len(chunks_mundial)}")

import statistics
for name, cks in [("plataforma", chunks_plataforma), ("mundial", chunks_mundial)]:
    sizes = [len(c["content"]) for c in cks]
    print(f"\n{name}: min={min(sizes)}, median={int(statistics.median(sizes))}, mean={int(statistics.mean(sizes))}, max={max(sizes)}")

Chunking en 0.0s

Chunks plataforma: 316
Chunks mundial:    4497
TOTAL: 4813

plataforma: min=40, median=214, mean=257, max=1177

mundial: min=41, median=1486, mean=1290, max=2597


In [4]:
# Inspeccion: primer chunk de cada coleccion
print("=== Plataforma — chunk 0 ===")
print("Metadata:", chunks_plataforma[0]["metadata"])
print("Content (primeros 300 chars):")
print(chunks_plataforma[0]["content"][:300])
print("\n=== Mundial — chunk 0 ===")
print("Metadata:", chunks_mundial[0]["metadata"])
print("Content (primeros 300 chars):")
print(chunks_mundial[0]["content"][:300])

=== Plataforma — chunk 0 ===
Metadata: {'titulo': 'Qué es 91', 'tema': 'identidad', 'tipo': 'concepto', 'fuente': 'General.txt', 'tags': 'overview, plataforma, mundial-2026, no-apuestas', 'coleccion': 'plataforma', 'ruta': 'Corpus_91/identidad/plataforma-overview.md', 'categoria': 'identidad', 'header_h1': 'Qué es 91', 'chunk_index': 0}
Content (primeros 300 chars):
# Qué es 91

91 es una plataforma digital para jugar el Mundial 2026 prediciendo resultados de partidos, compitiendo contra otros usuarios, formando tribus con amigos y participando en rankings y torneos.

La plataforma está pensada como una experiencia de predicción basada en conocimiento futbolero

=== Mundial — chunk 0 ===
Metadata: {'titulo': 'Mexico vs Sudafrica — Fase de Grupos, Grupo A', 'tema': 'calendario-mundial-2026', 'tipo': 'partido', 'fuente': 'Plataforma 91 / FIFA', 'partido_id': 1, 'edicion': 2026, 'fase': 'group', 'fase_es': 'Fase de Grupos', 'grupo': 'A', 'fecha': '2026-06-11', 'hora_local': '14:00', 'hora

## 4. Embeddings con OpenAI `text-embedding-3-small`

Usamos `langchain_openai.OpenAIEmbeddings`, que maneja batching y reintentos automaticamente. No necesitamos prefix (a diferencia de e5); el modelo es multilingual nativamente.

**Costo:** ~$0.02 por millon de tokens. Para ~4k chunks de ~500 tokens cada uno = ~2M tokens → **~$0.04 indexado completo**.

In [5]:
embedder = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    chunk_size=BATCH_SIZE_EMBED,  # tamaño del batch interno por request HTTP
)

# Test minimo (1 embedding) — confirma conexion y dim
_test = embedder.embed_query("hola mundo")
EMBED_DIM = len(_test)
print(f"Modelo OK — dim={EMBED_DIM} (esperado 1536)")


def embed_passages(texts):
    # Embeda lista de textos via OpenAI. langchain_openai maneja batching y reintentos.
    return embedder.embed_documents(texts)


def embed_queries(texts):
    # Embeda queries — para OpenAI no hay diferencia con passages (a diferencia de e5).
    return [embedder.embed_query(t) for t in texts]

Modelo OK — dim=1536 (esperado 1536)


## 5. Indexar en Chroma — colecciones persistentes

Dos colecciones independientes con `hnsw:space=cosine`. Se borran y recrean en cada Run All para garantizar idempotencia.

In [6]:
CHROMA_DIR.mkdir(exist_ok=True)
print(f"Chroma dir: {CHROMA_DIR}")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

for name in ["mundial", "plataforma"]:
    try:
        client.delete_collection(name)
        print(f"Coleccion previa '{name}' eliminada")
    except Exception:
        pass

col_mundial = client.create_collection(
    name="mundial",
    metadata={"hnsw:space": "cosine", "descripcion": "Corpus externo Mundial 2026 (Wikipedia + Reglamento + Kaggle)"},
)
col_plataforma = client.create_collection(
    name="plataforma",
    metadata={"hnsw:space": "cosine", "descripcion": "Corpus interno plataforma 91 (predicciones del usuario)"},
)
print(f"\nColecciones creadas: {[c.name for c in client.list_collections()]}")

Chroma dir: c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\chroma_db
Coleccion previa 'mundial' eliminada
Coleccion previa 'plataforma' eliminada

Colecciones creadas: ['plataforma', 'mundial']


In [7]:
def index_chunks(collection, chunks, coleccion_name):
    # Embeda via OpenAI y agrega a Chroma en batches.
    texts = [c["content"] for c in chunks]
    metas = [c["metadata"] for c in chunks]
    ids = [f"{coleccion_name}__{i:06d}" for i in range(len(chunks))]

    print(f"Embeddings {coleccion_name}: {len(texts)} chunks...")
    t0 = time.time()
    embs = embed_passages(texts)
    rate = len(texts) / max(time.time() - t0, 1e-6)
    print(f"  -> embed listo en {time.time() - t0:.1f}s ({rate:.0f} chunks/s)")

    print(f"Agregando a Chroma en batches de {BATCH_SIZE_CHROMA}...")
    t0 = time.time()
    for i in tqdm(range(0, len(texts), BATCH_SIZE_CHROMA)):
        j = i + BATCH_SIZE_CHROMA
        collection.add(
            ids=ids[i:j],
            documents=texts[i:j],
            metadatas=metas[i:j],
            embeddings=embs[i:j],
        )
    print(f"  -> indexado en {time.time() - t0:.1f}s")


index_chunks(col_plataforma, chunks_plataforma, "plataforma")
index_chunks(col_mundial, chunks_mundial, "mundial")

print(f"\nCount plataforma: {col_plataforma.count()}")
print(f"Count mundial:    {col_mundial.count()}")

Embeddings plataforma: 316 chunks...
  -> embed listo en 2.3s (135 chunks/s)
Agregando a Chroma en batches de 256...


  0%|          | 0/2 [00:00<?, ?it/s]

  -> indexado en 0.3s
Embeddings mundial: 4497 chunks...
  -> embed listo en 30.8s (146 chunks/s)
Agregando a Chroma en batches de 256...


  0%|          | 0/18 [00:00<?, ?it/s]

  -> indexado en 4.8s

Count plataforma: 316
Count mundial:    4497


## 6. Smoke test — retrieval cualitativo

Verificamos que el indice funciona corriendo 4 queries en cada coleccion y mostrando top-3 chunks con su score (1 − distancia coseno).

In [8]:
def retrieve(collection, query: str, k: int = 3) -> None:
    [q_emb] = embed_queries([query])
    res = collection.query(
        query_embeddings=[q_emb], n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    print(f"\nQ: {query!r}")
    for i, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0])):
        score = 1 - dist
        titulo = meta.get("titulo", "")
        cat = meta.get("categoria", "")
        preview = doc[:180].replace("\n", " ")
        print(f"  [{i+1}] score={score:.3f} | {cat}/{titulo}")
        print(f"       {preview}...")


print("=" * 60)
print("COLECCION MUNDIAL")
print("=" * 60)
for q in [
    "cuales son los 16 estadios del Mundial 2026",
    "quien fue campeon del Mundial 2022",
    "cuantos equipos participan en el Mundial 2026",
    "que dice el reglamento FIFA sobre el VAR",
]:
    retrieve(col_mundial, q, k=3)

print("\n" + "=" * 60)
print("COLECCION PLATAFORMA")
print("=" * 60)
for q in [
    "como crear una tribu en la plataforma",
    "cuantas monedas gano por invitar un amigo",
    "hasta cuando puedo editar mi prediccion",
    "que pasa si gano un torneo",
]:
    retrieve(col_plataforma, q, k=3)

COLECCION MUNDIAL

Q: 'cuales son los 16 estadios del Mundial 2026'
  [1] score=0.697 | wikipedia/Copa Mundial de Fútbol de 2026
       dos) y los gobiernos nacionales de cada país. Y por último, como encargados de los aspectos logísticos más esenciales, cada una de las 16 ciudades sede, estableció un «Comité del M...
  [2] score=0.669 | wikipedia/Copa Mundial de Fútbol de 2026
        angostas, solo requirieron adecuaciones mínimas para instalar césped natural y ampliar los espacios contiguos a la cancha, especialmente el Sofi Stadium. ​ En suma las 16 sedes se...
  [3] score=0.659 | wikipedia/Copa Mundial de Fútbol de 2026
       stadios que repita en la justa mundialista, ya que ninguno de los otros once estadios mexicanos sede en 1970 y 1986, y ninguno de los nueve estadios estadounidenses de 1994 volverá...

Q: 'quien fue campeon del Mundial 2022'
  [1] score=0.645 | kaggle/Copa Mundial de Fútbol de 2022
       # Copa Mundial de Fútbol de 2022  La Copa Mundial de Fútbol de 2022 se

## 7. Resumen final

In [9]:
print("=" * 60)
print("FASE 5 — INDICE VECTORIAL LISTO")
print("=" * 60)
print(f"Modelo embeddings: {EMBEDDING_MODEL} (dim={EMBED_DIM}, via OpenAI API)")
print(f"Chunk strategy: MarkdownHeaderTextSplitter + Recursive({CHUNK_SIZE}/{CHUNK_OVERLAP}) fallback")
print(f"Persistencia: {CHROMA_DIR}")
print()
print(f"Docs originales: {len(docs_plataforma) + len(docs_mundial)} (.md)")
print(f"  - plataforma: {len(docs_plataforma)}")
print(f"  - mundial:    {len(docs_mundial)}")
print()
print(f"Chunks indexados: {col_mundial.count() + col_plataforma.count()}")
print(f"  - plataforma: {col_plataforma.count()}")
print(f"  - mundial:    {col_mundial.count()}")
print()
print("Listo para F6 (agente Agentic RAG).")

FASE 5 — INDICE VECTORIAL LISTO
Modelo embeddings: text-embedding-3-small (dim=1536, via OpenAI API)
Chunk strategy: MarkdownHeaderTextSplitter + Recursive(2000/200) fallback
Persistencia: c:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\chroma_db

Docs originales: 493 (.md)
  - plataforma: 55
  - mundial:    438

Chunks indexados: 4813
  - plataforma: 316
  - mundial:    4497

Listo para F6 (agente Agentic RAG).
